# S6E9: sealed CV prototype

Public LB **0.94000超え**を目指す初回プロトタイプです。

コンペの `train.csv`, `test.csv`, `sample_submission.csv` をInputに追加してください。
`Yes=1` の確率を提出します。データ20%を探索・前処理から外し、候補固定後に一度評価します。

最初は各モデル2候補、1 seedです。2候補はrawとinteractionの比較で、Optunaの自由探索は3候補目からです。
探索を広げる場合は**封印評価の前に**設定します。既に封印結果を見た同じ分割は新たな独立評価には使えません。


In [1]:
from pathlib import Path
import importlib.util
import sys
import subprocess

required = ['numpy', 'pandas', 'sklearn', 'xgboost', 'lightgbm', 'catboost', 'optuna']
missing = [m for m in required if importlib.util.find_spec(m) is None]
print('Missing libraries:', missing)
# 不足があればInternetを有効にして次を実行し、必要に応じてセッションを再起動:
# subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'scikit-learn>=1.4,<2',
#     'xgboost>=2.1,<4', 'lightgbm>=4.3,<5', 'catboost>=1.2.7,<2', 'optuna>=4,<6'])
assert not missing, 'Install the missing libraries first.'


Missing libraries: []


In [2]:
from pathlib import Path
import csv


def _is_s6e9_data(folder):
    required = ('train.csv', 'test.csv', 'sample_submission.csv')
    try:
        headers = []
        for filename in required:
            with (folder / filename).open(encoding='utf-8-sig', newline='') as stream:
                headers.append(next(csv.reader(stream)))
        train, test, sample = headers
        if any(len(h) != len(set(h)) for h in headers):
            return False
        target = 'Will_Buy_EV'
        return (set(sample) == {'id', target}
                and {'id', target, 'Age', 'Annual_Income_USD', 'Daily_Commute_km'} <= set(train)
                and target not in test and set(train) - {target} == set(test))
    except (OSError, UnicodeError, StopIteration, csv.Error):
        return False


def find_s6e9_data(input_root='/kaggle/input', override=None):
    root = Path(input_root)
    if override is not None:
        folder = Path(override)
        if _is_s6e9_data(folder):
            return folder
        raise FileNotFoundError(
            f'DATA_OVERRIDE={folder} に、S6E9の train.csv / test.csv / sample_submission.csv '
            'が揃っているか、CSVの列名を確認してください。')
    for folder in (root / 'competitions/playground-series-s6e9', root / 'playground-series-s6e9'):
        if _is_s6e9_data(folder):
            return folder
    matches = sorted({p.parent for p in root.rglob('train.csv') if _is_s6e9_data(p.parent)}) if root.is_dir() else []
    if len(matches) == 1:
        return matches[0]
    if len(matches) > 1:
        raise ValueError('対応するデータが複数あります。DATA_OVERRIDE に使うフォルダーを指定してください:\n'
                         + '\n'.join(str(p) for p in matches))
    available = '\n'.join(f'  - {p.name}' for p in sorted(root.iterdir())[:20]) if root.is_dir() else '  （入力なし）'
    raise FileNotFoundError(
        'S6E9の学習データが見つかりません。\n'
        '右側の Add Input からコンペ本体「Predicting Electric Vehicle Purchases」'
        '（playground-series-s6e9）を追加し、このセルを再実行してください。\n'
        '必要なファイル: train.csv / test.csv / sample_submission.csv\n'
        '0.946xx.csv などの提出予測CSVだけのデータセットでは学習できません。\n'
        'ZIPをUploadした場合は、CSVを展開したフォルダーを DATA_OVERRIDE に指定してください。\n'
        f'現在の入力:\n{available}')


DATA_OVERRIDE = None  # 自動検出できない場合だけ、CSVが3つ揃ったフォルダーを指定
DATA = find_s6e9_data(override=DATA_OVERRIDE)
RUN = Path('/kaggle/working/s6e9_run')
TRIALS_PER_FAMILY = 2    # 8 or 30: more Optuna exploration, before audit only
SEEDS = [2026]          # [2026, 42, 3407]: seed averaging
MAX_ROUNDS = 1600       # 10000 for a larger search
EARLY_STOPPING = 100    # 200 for a larger search
THREADS = 4
XGB_DEVICE = 'cpu'      # 'cuda' when a compatible GPU accelerator is enabled
FAMILIES = ['lgb', 'xgb']  # Add 'cat' for the more expensive CatBoost experiments
print('Data:', DATA, 'Outputs:', RUN)
print('Input files:', [p.name for p in DATA.iterdir() if p.is_file()])


Data: /kaggle/input/competitions/playground-series-s6e9 Outputs: /kaggle/working/s6e9_run
Input files: ['sample_submission.csv', 'train.csv', 'test.csv']


## 実装を作業領域に展開
このNotebookは学習スクリプトを内包しているので、別途 `.py` のInput追加はしません。

In [3]:
SCRIPT = Path('/kaggle/working/prototype.py')
SCRIPT.write_text('"""S6E9: development-only search, frozen sealed audit, final 5-fold ensemble."""\nfrom __future__ import annotations\n\nimport argparse\nimport hashlib\nimport importlib.metadata\nimport json\nimport os\nfrom pathlib import Path\nimport platform\nimport re\nimport time\n\nimport numpy as np\nimport pandas as pd\nfrom sklearn.metrics import roc_auc_score\nfrom sklearn.model_selection import KFold, StratifiedKFold, train_test_split\n\nTARGET = "Will_Buy_EV"\nSEED = 2026\n\n\ndef write_json(path, obj):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    tmp = path.with_suffix(path.suffix + ".tmp")\n    tmp.write_text(json.dumps(obj, indent=2, ensure_ascii=False, allow_nan=False), encoding="utf-8")\n    tmp.replace(path)\n\n\ndef read_json(path):\n    return json.loads(Path(path).read_text(encoding="utf-8"))\n\n\ndef digest(path):\n    h = hashlib.sha256()\n    with open(path, "rb") as f:\n        for block in iter(lambda: f.read(1024 * 1024), b""):\n            h.update(block)\n    return h.hexdigest()\n\n\ndef load_data(data):\n    data = Path(data)\n    required = ["train.csv", "test.csv", "sample_submission.csv"]\n    if not all((data / f).is_file() for f in required):\n        raise FileNotFoundError(f"Place {required} in {data}; no artificial fallback is used.")\n    tr, te, sub = [pd.read_csv(data / f) for f in required]\n    if TARGET not in tr or TARGET in te or TARGET not in sub:\n        raise ValueError(f"Expected binary target {TARGET}")\n    ids = [c for c in sub if c != TARGET]\n    if len(ids) != 1:\n        raise ValueError("Expected one submission ID column")\n    id_col = ids[0]\n    if set(tr[TARGET].unique()) == {"No", "Yes"}:\n        tr[TARGET] = tr[TARGET].map({"No": 0, "Yes": 1})\n    if not set(tr[TARGET].unique()) == {0, 1}:\n        raise ValueError("Target must contain exactly 0 and 1, with no missing labels")\n    for frame in (tr, te, sub):\n        if id_col not in frame or frame[id_col].isna().any() or frame[id_col].duplicated().any():\n            raise ValueError("Missing or duplicate IDs")\n    if set(te[id_col]) != set(sub[id_col]) or len(te) != len(sub):\n        raise ValueError("Test/submission IDs differ")\n    if set(tr[id_col]) & set(te[id_col]):\n        raise ValueError("Train/test IDs overlap")\n    features = [c for c in tr if c not in (id_col, TARGET)]\n    if set(features) != set(te.columns) - {id_col}:\n        raise ValueError("Train/test feature schemas differ")\n    return tr, te, sub, id_col, features\n\n\ndef split_plan(y, seed=SEED):\n    """Sealed fold never enters any development fold\'s training indices."""\n    outer = np.full(len(y), -1, dtype=int)\n    for fold, (_, va) in enumerate(StratifiedKFold(5, shuffle=True, random_state=seed).split(np.zeros(len(y)), y)):\n        outer[va] = fold\n    dev, sealed = np.flatnonzero(outer != 4), np.flatnonzero(outer == 4)\n    cv = [(dev[tr], dev[va]) for tr, va in StratifiedKFold(4, shuffle=True, random_state=seed + 1).split(dev, y[dev])]\n    return dev, sealed, cv, outer\n\n\ndef augment(x, variant):\n    x = x.copy().reset_index(drop=True)\n    # Stable internal names work with all three libraries, including LightGBM.\n    original = list(x)\n    x.columns = [f"f{i}" for i in range(len(original))]\n    if variant in ("interaction", "artifact"):\n        numeric = [c for c in x if pd.api.types.is_numeric_dtype(x[c])]\n        for c in numeric:\n            values = x[c].astype(float)\n            if variant == "interaction":\n                x[c + "_log"] = np.sign(values) * np.log1p(np.abs(values))\n            else:\n                x[c + "_fraction"] = values - np.floor(values)\n                x[c + "_rounded"] = values.round(1)\n        lookup = {re.sub(r"[^a-z0-9]", "", c.lower()): f"f{i}" for i, c in enumerate(original)}\n        pairs = [("Annual_Income", "Vehicle_Cost"), ("Monthly_Income", "Vehicle_Cost"),\n                 ("Daily_Commute_Distance", "Charging_Stations_Nearby"),\n                 ("Daily_Usage_km", "Battery_Range_km")]\n        for a, b in pairs:\n            ca, cb = [lookup.get(re.sub(r"[^a-z0-9]", "", c.lower())) for c in (a, b)]\n            if ca in numeric and cb in numeric:\n                x[f"{ca}_over_{cb}"] = x[ca] / (x[cb].abs() + 1)\n        cats = [c for c in x if not pd.api.types.is_numeric_dtype(x[c])]\n        # A bounded generic interaction set; variant must earn its place in dev CV.\n        for a, b in zip(cats[:4], cats[1:5]):\n            x[f"{a}_cross_{b}"] = x[a].astype("string").fillna("<NA>") + "|" + x[b].astype("string").fillna("<NA>")\n        # S6E9 domain features: deterministic, no target or validation statistics.\n        names = {c: f"f{i}" for i, c in enumerate(original)}\n        home, work = names.get("Charging_Stations_Near_Home"), names.get("Charging_Stations_Near_Work")\n        commute = names.get("Daily_Commute_km")\n        if home is not None and work is not None:\n            x["stations_total"] = x[home] + x[work]\n            x["stations_gap"] = x[home] - x[work]\n            if commute is not None:\n                x["commute_per_station"] = x[commute] / (1 + x["stations_total"])\n        anxiety = names.get("Range_Anxiety_Level")\n        if anxiety is not None:\n            x["anxiety_ordinal"] = x[anxiety].map({"Low": 0, "Medium": 1, "High": 2})\n        income, cars = names.get("Annual_Income_USD"), names.get("Number_of_Cars_Owned")\n        if income is not None and cars is not None:\n            x["income_per_car"] = x[income] / (1 + x[cars])\n    return x\n\n\nclass Features:\n    """All learned mappings fit on this fit partition only, including frequency/TE."""\n    def __init__(self, variant="raw", seed=SEED):\n        self.variant, self.seed = variant, seed\n\n    @staticmethod\n    def tokens(s):\n        return s.astype("string").fillna("<MISSING>")\n\n    def te_map(self, s, y):\n        stats = pd.DataFrame({"key": s.to_numpy(), "y": np.asarray(y)}).groupby("key")["y"].agg(["sum", "count"])\n        prior = float(np.mean(y))\n        return (stats["sum"] + 20 * prior) / (stats["count"] + 20), prior\n\n    def fit_transform(self, x, y):\n        z = augment(x, self.variant)\n        self.cats = [c for c in z if not pd.api.types.is_numeric_dtype(z[c])]\n        self.maps = {c: {v: i + 1 for i, v in enumerate(sorted(self.tokens(z[c]).unique()))} for c in self.cats}\n        self.freq_cols = list(z) if self.variant in ("frequency", "artifact") else []\n        self.freq = {c: self.tokens(z[c]).value_counts(normalize=True) for c in self.freq_cols}\n        self.te = {c: self.te_map(self.tokens(z[c]), y) for c in self.cats} if self.variant == "target" else {}\n        result = self.transform(x)\n        # KFold assignment does not depend on labels. Both map AND prior exclude each row.\n        if self.te:\n            for c in self.cats:\n                values = np.empty(len(z), dtype=np.float32)\n                for tr, va in KFold(4, shuffle=True, random_state=self.seed).split(z):\n                    mapping, prior = self.te_map(self.tokens(z[c]).iloc[tr], np.asarray(y)[tr])\n                    values[va] = self.tokens(z[c]).iloc[va].map(mapping).fillna(prior)\n                result[c + "_te"] = values\n        return result\n\n    def transform(self, x):\n        z = augment(x, self.variant)\n        out = z.copy()\n        for c in self.cats:\n            codes = self.tokens(z[c]).map(self.maps[c]).fillna(0).astype(int)\n            out[c] = pd.Categorical(codes, categories=range(len(self.maps[c]) + 1))\n        for c in self.freq_cols:\n            out[c + "_freq"] = self.tokens(z[c]).map(self.freq[c]).fillna(0).astype(np.float32)\n        for c, (mapping, prior) in self.te.items():\n            out[c + "_te"] = self.tokens(z[c]).map(mapping).fillna(prior).astype(np.float32)\n        for c in out:\n            if c not in self.cats:\n                out[c] = pd.to_numeric(out[c], errors="coerce").replace([np.inf, -np.inf], np.nan).astype(np.float32)\n        return out\n\n\ndef model_frame(x, family):\n    if family != "cat":\n        return x\n    x = x.copy()\n    for c in x.select_dtypes(include="category"):\n        x[c] = x[c].astype(int).astype(str)\n    return x\n\n\ndef fit_model(family, params, x, y, valid, seed, cfg, rounds=None):\n    n = int(rounds or cfg["max_rounds"])\n    x = model_frame(x, family)\n    valid = None if valid is None else (model_frame(valid[0], family), valid[1])\n    stop = cfg["early_stopping"]\n    if family == "xgb":\n        from xgboost import XGBClassifier\n        model = XGBClassifier(**params, n_estimators=n, objective="binary:logistic", eval_metric="auc",\n            tree_method="hist", enable_categorical=True, device=cfg["xgb_device"],\n            n_jobs=cfg["threads"], random_state=seed, early_stopping_rounds=stop if valid else None)\n        model.fit(x, y, eval_set=[valid] if valid else None, verbose=False)\n        best = model.best_iteration + 1 if valid else n\n    elif family == "lgb":\n        import lightgbm as lgb\n        model = lgb.LGBMClassifier(**params, n_estimators=n, objective="binary", metric="auc",\n            n_jobs=cfg["threads"], random_state=seed, verbosity=-1, deterministic=True, force_col_wise=True)\n        model.fit(x, y, eval_set=[valid] if valid else None,\n            callbacks=[lgb.early_stopping(stop, verbose=False)] if valid else [])\n        best = model.best_iteration_ if valid else n\n    else:\n        from catboost import CatBoostClassifier\n        cats = list(x.select_dtypes(include=["object", "string"]))\n        model = CatBoostClassifier(**params, iterations=n, loss_function="Logloss", eval_metric="AUC",\n            thread_count=cfg["threads"], random_seed=seed, allow_writing_files=False, verbose=False)\n        model.fit(x, y, cat_features=cats, eval_set=valid, use_best_model=bool(valid),\n            early_stopping_rounds=stop if valid else None, verbose=False)\n        best = model.get_best_iteration() + 1 if valid else n\n    return model, max(1, int(best))\n\n\ndef predict(model, x, family):\n    p = model.predict_proba(model_frame(x, family))[:, 1]\n    if not np.isfinite(p).all() or ((p < 0) | (p > 1)).any():\n        raise ValueError("Invalid probabilities")\n    return p\n\n\ndef defaults(family):\n    if family == "xgb":\n        return dict(max_depth=5, min_child_weight=8., gamma=0.1, reg_alpha=0.01,\n            reg_lambda=8., subsample=0.85, colsample_bytree=0.85, learning_rate=0.04)\n    if family == "lgb":\n        return dict(max_depth=6, num_leaves=31, min_child_samples=80, reg_alpha=0.01,\n            reg_lambda=8., subsample=0.85, subsample_freq=1, colsample_bytree=0.85, learning_rate=0.04)\n    return dict(depth=6, learning_rate=0.04, l2_leaf_reg=8., random_strength=1.,\n        bootstrap_type="Bayesian", bagging_temperature=1.)\n\n\ndef suggest(trial, family):\n    p = defaults(family)\n    p["learning_rate"] = trial.suggest_float("learning_rate", 0.01, 0.1, log=True)\n    if family in ("xgb", "lgb"):\n        p.update(max_depth=trial.suggest_int("max_depth", 3, 9),\n            reg_alpha=trial.suggest_float("reg_alpha", 1e-4, 10, log=True),\n            reg_lambda=trial.suggest_float("reg_lambda", 1e-2, 30, log=True),\n            subsample=trial.suggest_float("subsample", 0.65, 1),\n            colsample_bytree=trial.suggest_float("colsample_bytree", 0.6, 1))\n        if family == "xgb":\n            p.update(min_child_weight=trial.suggest_float("min_child_weight", 1, 30, log=True),\n                     gamma=trial.suggest_float("gamma", 0, 5))\n        else:\n            p.update(num_leaves=min(2 ** p["max_depth"], trial.suggest_int("num_leaves", 15, 127)),\n                     min_child_samples=trial.suggest_int("min_child_samples", 40, 400))\n    else:\n        p.update(depth=trial.suggest_int("depth", 4, 8), l2_leaf_reg=trial.suggest_float("l2_leaf_reg", 1, 30, log=True),\n            random_strength=trial.suggest_float("random_strength", 0.01, 5, log=True),\n            bagging_temperature=trial.suggest_float("bagging_temperature", 0, 5))\n    return p\n\n\ndef context(args, create=False):\n    tr, te, sub, id_col, features = load_data(args.data)\n    run = Path(args.run)\n    cfg_path = run / "config.json"\n    hashes = {name: digest(Path(args.data) / name) for name in ("train.csv", "test.csv", "sample_submission.csv")}\n    code_hash = digest(__file__)\n    if create and not cfg_path.exists():\n        cfg = dict(seed=SEED, max_rounds=args.max_rounds, early_stopping=args.early_stopping,\n            threads=args.threads, xgb_device=args.xgb_device, seeds=args.seeds,\n            hashes=hashes, code_sha256=code_hash, families=args.families,\n            versions={p: importlib.metadata.version(p) for p in ("numpy", "pandas", "scikit-learn", "xgboost", "lightgbm", "catboost", "optuna")},\n            python=platform.python_version())\n        write_json(cfg_path, cfg)\n    cfg = read_json(cfg_path)\n    if cfg["hashes"] != hashes or cfg["code_sha256"] != code_hash:\n        raise ValueError("Data/code changed. Use a fresh run directory; do not reuse a revealed sealed holdout for tuning.")\n    if create:\n        for key in ("max_rounds", "early_stopping", "threads", "xgb_device", "seeds", "families"):\n            if getattr(args, key) != cfg[key]:\n                raise ValueError(f"Resume configuration changed: {key}")\n    y = tr[TARGET].to_numpy(dtype=int)\n    dev, sealed, cv, outer = split_plan(y, cfg["seed"])\n    return tr, te, sub, id_col, features, run, cfg, y, dev, sealed, cv, outer\n\n\ndef run_cv(x, y, cv, candidate, cfg, seeds):\n    oof = np.full(len(y), np.nan)\n    rounds, scores = [], []\n    family, variant, params = [candidate[k] for k in ("family", "variant", "params")]\n    for fold, (it, iv) in enumerate(cv):\n        preds = []\n        for seed in seeds:\n            fe = Features(variant, seed)\n            xt = fe.fit_transform(x.iloc[it], y[it])\n            xv = fe.transform(x.iloc[iv])\n            model, n = fit_model(family, params, xt, y[it], (xv, y[iv]), seed, cfg)\n            preds.append(predict(model, xv, family))\n            rounds.append(n)\n        oof[iv] = np.mean(preds, axis=0)\n        scores.append(float(roc_auc_score(y[iv], oof[iv])))\n        print(f"  {family}/{variant} fold={fold} AUC={scores[-1]:.6f}", flush=True)\n    return oof, rounds, scores\n\n\ndef search(args):\n    tr, te, sub, id_col, cols, run, cfg, y, dev, sealed, cv, outer = context(args, True)\n    if (run / "frozen.json").exists() or (run / "SEALED_OPENED.json").exists():\n        raise RuntimeError("This experiment is frozen; search is disabled.")\n    pd.DataFrame({id_col: tr[id_col], "outer_fold": outer}).to_csv(run / "folds.csv", index=False)\n    import optuna\n    for family in cfg["families"]:\n        study = optuna.create_study(study_name=family, storage=f"sqlite:///{(run / \'optuna.db\').resolve().as_posix()}",\n            load_if_exists=True, direction="maximize", sampler=optuna.samplers.TPESampler(seed=cfg["seed"]))\n        # Every family gets a raw baseline even when only one trial is requested.\n        if not study.trials:\n            for variant in ("raw", "interaction"):\n                study.enqueue_trial({"variant": variant, **{k: v for k, v in defaults(family).items() if k not in ("subsample_freq", "bootstrap_type")}})\n        study.sampler = optuna.samplers.TPESampler(seed=cfg["seed"] + len(study.trials))\n        def objective(trial):\n            started = time.time()\n            variant = trial.suggest_categorical("variant", ["raw", "frequency", "interaction", "artifact", "target"])\n            candidate = dict(family=family, variant=variant, params=suggest(trial, family), trial=trial.number)\n            oof, rounds, scores = run_cv(tr[cols], y, cv, candidate, cfg, [cfg["seed"]])\n            name = f"{family}_{trial.number}"\n            np.save(run / f"{name}_oof.npy", oof)\n            candidate.update(rounds=rounds, fold_auc=scores, dev_auc=float(roc_auc_score(y[dev], oof[dev])), seconds=time.time() - started)\n            write_json(run / f"{name}.json", candidate)\n            trial.set_user_attr("artifact", name)\n            return candidate["dev_auc"]\n        # trials is a TOTAL completed-trial budget per family, making resume idempotent.\n        completed = len([t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE])\n        study.optimize(objective, n_trials=max(0, args.trials - completed))\n        study.trials_dataframe().to_csv(run / f"{family}_trials.csv", index=False)\n    print("Search complete. Sealed scores have NOT been computed.", flush=True)\n\n\ndef blend_weights(y, matrix):\n    # Small greedy convex search: 20 steps, no unconstrained optimizer.\n    total = np.zeros(len(y))\n    counts = np.zeros(matrix.shape[1])\n    best_score, best = -1, None\n    for step in range(20):\n        scores = [roc_auc_score(y, (total + matrix[:, j]) / (step + 1)) for j in range(matrix.shape[1])]\n        j = int(np.argmax(scores))\n        counts[j] += 1\n        total += matrix[:, j]\n        if scores[j] > best_score:\n            best_score, best = float(scores[j]), counts.copy() / (step + 1)\n    return best, best_score\n\n\ndef freeze(args):\n    tr, te, sub, id_col, cols, run, cfg, y, dev, sealed, cv, outer = context(args)\n    if (run / "frozen.json").exists():\n        print("Already frozen; no changes made.")\n        return\n    if (run / "SEALED_OPENED.json").exists():\n        raise RuntimeError("Sealed fold already opened")\n    candidates = []\n    for family in cfg["families"]:\n        records = [read_json(p) for p in run.glob(f"{family}_[0-9]*.json")]\n        if not records:\n            raise RuntimeError(f"Run search for {family} first")\n        candidates.append(max(records, key=lambda c: c["dev_auc"]))\n    # A fixed, reproducible comparator, independent of the search winners.\n    baseline = dict(family="lgb", variant="raw", params=defaults("lgb"), trial=-1)\n    all_candidates = candidates + [baseline]\n    matrix = []\n    for i, c in enumerate(all_candidates):\n        cached = run / f"{c[\'family\']}_{c[\'trial\']}_oof.npy"\n        if c["trial"] == -1:\n            record_path = run / "lgb_0.json"\n            if record_path.exists():\n                record = read_json(record_path)\n                if record["variant"] == "raw" and record["params"] == c["params"]:\n                    cached = run / "lgb_0_oof.npy"\n                    c["rounds"] = record["rounds"]\n                    c["fold_auc"] = record["fold_auc"]\n        if cfg["seeds"] == [cfg["seed"]] and cached.exists():\n            oof, rounds, scores = np.load(cached), c["rounds"], c["fold_auc"]\n        else:\n            oof, rounds, scores = run_cv(tr[cols], y, cv, c, cfg, cfg["seeds"])\n        c["fixed_rounds"] = int(np.median(rounds))\n        c["seed_dev_auc"] = float(roc_auc_score(y[dev], oof[dev]))\n        c["seed_fold_auc"] = scores\n        matrix.append(oof[dev])\n        np.save(run / f"frozen_candidate_{i}_dev_oof.npy", oof)\n    matrix = np.column_stack(matrix)\n    weights, score = blend_weights(y[dev], matrix[:, :-1])\n    pd.DataFrame(matrix, columns=[f"candidate_{i}" for i in range(len(all_candidates))]).corr().to_csv(run / "dev_prediction_correlation.csv")\n    report = pd.DataFrame({id_col: tr[id_col].iloc[dev].to_numpy(), TARGET: y[dev]})\n    for i in range(matrix.shape[1]):\n        report[f"candidate_{i}"] = matrix[:, i]\n    report["blend"] = matrix[:, :-1] @ weights\n    report.to_csv(run / "dev_oof.csv", index=False)\n    write_json(run / "frozen.json", dict(candidates=candidates, baseline=baseline,\n        weights=weights.tolist(), development_selection_auc=score,\n        warning="Development AUC is selection-biased; sealed audit is the independent check.",\n        config_sha256=digest(run / "config.json")))\n    print(f"Frozen dev selection AUC={score:.6f}; weights={weights}. Next: audit.", flush=True)\n\n\ndef fitted_predictions(x, y, xpred, candidate, cfg, model_dir=None):\n    values = []\n    family = candidate["family"]\n    for seed in cfg["seeds"]:\n        fe = Features(candidate["variant"], seed)\n        xt = fe.fit_transform(x, y)\n        xp = fe.transform(xpred)\n        model, _ = fit_model(family, candidate["params"], xt, y, None, seed, cfg, candidate["fixed_rounds"])\n        values.append(predict(model, xp, family))\n        if model_dir is not None:\n            import joblib\n            Path(model_dir).mkdir(parents=True, exist_ok=True)\n            joblib.dump({"features": fe, "model": model, "family": family}, Path(model_dir) / f"seed_{seed}.joblib")\n    return np.mean(values, axis=0)\n\n\ndef paired_bootstrap(y, p, baseline, n=300):\n    rng = np.random.default_rng(SEED)\n    pos, neg = np.flatnonzero(y == 1), np.flatnonzero(y == 0)\n    aucs, deltas = [], []\n    for _ in range(n):\n        ix = np.concatenate([rng.choice(pos, len(pos)), rng.choice(neg, len(neg))])\n        a = roc_auc_score(y[ix], p[ix])\n        aucs.append(a)\n        deltas.append(a - roc_auc_score(y[ix], baseline[ix]))\n    return dict(auc_95_ci=np.quantile(aucs, [0.025, 0.975]).tolist(),\n                delta_95_ci=np.quantile(deltas, [0.025, 0.975]).tolist(), bootstrap_replicates=n)\n\n\ndef audit(args):\n    tr, te, sub, id_col, cols, run, cfg, y, dev, sealed, cv, outer = context(args)\n    frozen = read_json(run / "frozen.json")\n    if frozen["config_sha256"] != digest(run / "config.json"):\n        raise ValueError("Frozen configuration mismatch")\n    mark = run / "SEALED_OPENED.json"\n    if mark.exists() and read_json(mark)["frozen_sha256"] != digest(run / "frozen.json"):\n        raise ValueError("Frozen candidates changed after audit started")\n    if (run / "sealed_report.json").exists():\n        print(json.dumps(read_json(run / "sealed_report.json"), indent=2))\n        return\n    write_json(mark, {"frozen_sha256": digest(run / "frozen.json"), "opened_at": time.time()})\n    # No sealed labels are supplied to fitting, encoding, early stopping or weight selection.\n    preds = [fitted_predictions(tr[cols].iloc[dev], y[dev], tr[cols].iloc[sealed], c, cfg) for c in frozen["candidates"]]\n    p = np.column_stack(preds) @ np.asarray(frozen["weights"])\n    baseline = fitted_predictions(tr[cols].iloc[dev], y[dev], tr[cols].iloc[sealed], frozen["baseline"], cfg)\n    report = dict(sealed_auc=float(roc_auc_score(y[sealed], p)),\n        baseline_sealed_auc=float(roc_auc_score(y[sealed], baseline)), public_lb=None,\n        public_lb_target=0.94635, sealed_rows=len(sealed),\n        note="Public LB is unmeasured. Do not tune using this sealed result.")\n    report["delta_vs_baseline"] = report["sealed_auc"] - report["baseline_sealed_auc"]\n    report.update(paired_bootstrap(y[sealed], p, baseline))\n    pd.DataFrame({id_col: tr[id_col].iloc[sealed].to_numpy(), TARGET: y[sealed], "prediction": p, "baseline": baseline}).to_csv(run / "sealed_predictions.csv", index=False)\n    write_json(run / "sealed_report.json", report)\n    print(json.dumps(report, indent=2), flush=True)\n\n\ndef finalize(args):\n    tr, te, sub, id_col, cols, run, cfg, y, dev, sealed, cv, outer = context(args)\n    frozen = read_json(run / "frozen.json")\n    if not (run / "sealed_report.json").exists():\n        raise RuntimeError("Run the frozen sealed audit first")\n    if read_json(run / "SEALED_OPENED.json")["frozen_sha256"] != digest(run / "frozen.json"):\n        raise ValueError("Candidates changed after audit")\n    all_oof, all_test = [], []\n    for ci, (c, weight) in enumerate(zip(frozen["candidates"], frozen["weights"])):\n        if weight == 0:\n            continue\n        oof, test_preds = np.empty(len(y)), []\n        for fold in range(5):\n            it, iv = np.flatnonzero(outer != fold), np.flatnonzero(outer == fold)\n            # Fixed rounds from development CV: no final-fold early stopping or retuning.\n            both = pd.concat([tr[cols].iloc[iv], te[cols]], ignore_index=True)\n            p = fitted_predictions(tr[cols].iloc[it], y[it], both, c, cfg,\n                run / "models" / f"candidate_{ci}_fold_{fold}" if args.save_models else None)\n            oof[iv] = p[:len(iv)]\n            test_preds.append(p[len(iv):])\n            print(f"Final candidate={ci}, fold={fold} done", flush=True)\n        test_pred = np.mean(test_preds, axis=0)\n        all_oof.append(oof * weight)\n        all_test.append(test_pred * weight)\n        np.savez_compressed(run / f"final_candidate_{ci}.npz", oof=oof, test=test_pred)\n    poof, ptest = np.sum(all_oof, axis=0), np.sum(all_test, axis=0)\n    # Align to sample_submission by ID, never assume its row order equals test order.\n    mapped = pd.Series(ptest, index=te[id_col])\n    sub[TARGET] = sub[id_col].map(mapped)\n    if sub[TARGET].isna().any() or not sub[TARGET].between(0, 1).all():\n        raise ValueError("Invalid submission")\n    sub.to_csv(run / "submission.csv", index=False)\n    pd.DataFrame({id_col: tr[id_col], TARGET: y, "fold": outer, "prediction": poof}).to_csv(run / "final_oof.csv", index=False)\n    write_json(run / "final_report.json", dict(final_oof_auc=float(roc_auc_score(y, poof)), public_lb=None,\n        note="Final OOF reuses development-selected configurations and is NOT an independent score estimate.",\n        submission_rows=len(sub), submission_sha256=digest(run / "submission.csv")))\n    print(f"Created {run / \'submission.csv\'}; Public LB remains unmeasured.", flush=True)\n\n\ndef diagnose(args):\n    tr, te, sub, id_col, cols = load_data(args.data)\n    y = tr[TARGET].to_numpy(dtype=int)\n    dev, _, _, _ = split_plan(y)\n    a = tr.iloc[dev][cols].sample(min(30000, len(dev)), random_state=SEED)\n    b = te[cols].sample(min(30000, len(te)), random_state=SEED)\n    x = pd.concat([a, b], ignore_index=True)\n    labels = np.r_[np.zeros(len(a), int), np.ones(len(b), int)]\n    it, iv = train_test_split(np.arange(len(x)), test_size=0.3, stratify=labels, random_state=SEED)\n    fe = Features("raw")\n    xt = fe.fit_transform(x.iloc[it], labels[it])\n    xv = fe.transform(x.iloc[iv])\n    cfg = dict(max_rounds=300, early_stopping=40, threads=args.threads, xgb_device="cpu")\n    # Fixed complexity to avoid optimizing/reporting the same AV holdout.\n    model, _ = fit_model("lgb", defaults("lgb"), xt, labels[it], None, SEED, cfg, 300)\n    p = predict(model, xv, "lgb")\n    report = dict(adversarial_auc=float(roc_auc_score(labels[iv], p)), id_excluded=True,\n        sealed_excluded=True, train_rows=len(tr), test_rows=len(te), features=cols,\n        development_positive_rate=float(y[dev].mean()),\n        development_duplicate_feature_rows=int(tr.iloc[dev][cols].duplicated().sum()),\n        test_duplicate_feature_rows=int(te[cols].duplicated().sum()),\n        note="AV near 0.5 means this classifier found little shift; it does not prove identical distributions.")\n    Path(args.run).mkdir(parents=True, exist_ok=True)\n    write_json(Path(args.run) / "diagnostics.json", report)\n    pd.DataFrame({"feature": cols, "importance": model.feature_importances_}).sort_values("importance", ascending=False).to_csv(Path(args.run) / "adversarial_importance.csv", index=False)\n    pd.DataFrame({"dev_missing": tr.iloc[dev][cols].isna().mean(), "test_missing": te[cols].isna().mean(),\n        "dev_unique": tr.iloc[dev][cols].nunique(), "test_unique": te[cols].nunique()}).to_csv(Path(args.run) / "schema_diagnostics.csv")\n    print(json.dumps(report, indent=2), flush=True)\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("command", choices=["diagnose", "search", "freeze", "audit", "finalize"])\n    parser.add_argument("--data", default="/kaggle/input/competitions/playground-series-s6e9")\n    parser.add_argument("--run", default="runs/prototype")\n    parser.add_argument("--trials", type=int, default=8, help="Total completed trials per family; increase to resume")\n    parser.add_argument("--families", nargs="+", choices=["xgb", "cat", "lgb"], default=["xgb", "cat", "lgb"])\n    parser.add_argument("--max-rounds", type=int, default=3000)\n    parser.add_argument("--early-stopping", type=int, default=150)\n    parser.add_argument("--threads", type=int, default=min(4, os.cpu_count() or 1))\n    parser.add_argument("--xgb-device", choices=["cpu", "cuda"], default="cpu")\n    parser.add_argument("--seeds", nargs="+", type=int, default=[2026, 42])\n    parser.add_argument("--save-models", action="store_true")\n    args = parser.parse_args()\n    if min(args.trials, args.max_rounds, args.early_stopping, args.threads) < 1:\n        parser.error("Numeric budgets must be positive")\n    if len(set(args.seeds)) != len(args.seeds) or len(set(args.families)) != len(args.families):\n        parser.error("Duplicate seeds/families")\n    globals()[args.command](args)\n\n\nif __name__ == "__main__":\n    main()\n', encoding='utf-8')
print('Training script ready:', SCRIPT)


Training script ready: /kaggle/working/prototype.py


In [4]:
def execute(command, extra=()):
    args = [sys.executable, '-u', str(SCRIPT), command, '--data', str(DATA), '--run', str(RUN), *extra]
    subprocess.run(args, check=True)

execute('diagnose', ['--threads', str(THREADS)])


{
  "adversarial_auc": 0.4930258148148149,
  "id_excluded": true,
  "sealed_excluded": true,
  "train_rows": 668665,
  "test_rows": 286571,
  "features": [
    "Age",
    "Annual_Income_USD",
    "Daily_Commute_km",
    "Number_of_Cars_Owned",
    "Charging_Stations_Near_Home",
    "Charging_Stations_Near_Work",
    "Environmental_Concern_Level",
    "Gender",
    "City_Type",
    "Current_Car_Type",
    "Home_Charging_Possible",
    "Subsidy_Available",
    "Range_Anxiety_Level"
  ],
  "development_positive_rate": 0.17464462772838416,
  "development_duplicate_feature_rows": 0,
  "test_duplicate_feature_rows": 0,
  "note": "AV near 0.5 means this classifier found little shift; it does not prove identical distributions."
}


## developmentだけで探索
封印ラベルはスコア計算・early stopping・エンコーディングに使いません。

In [5]:
execute('search', ['--trials', str(TRIALS_PER_FAMILY), '--max-rounds', str(MAX_ROUNDS),
    '--early-stopping', str(EARLY_STOPPING), '--threads', str(THREADS), '--xgb-device', XGB_DEVICE,
    '--seeds', *map(str, SEEDS), '--families', *FAMILIES])


[I 2026-09-09 15:05:21,221] A new study created in RDB with name: lgb


  lgb/raw fold=0 AUC=0.942001
  lgb/raw fold=1 AUC=0.940933
  lgb/raw fold=2 AUC=0.941902
  lgb/raw fold=3 AUC=0.942193


[I 2026-09-09 15:07:20,444] Trial 0 finished with value: 0.9417468412110894 and parameters: {'variant': 'raw', 'learning_rate': 0.04, 'max_depth': 6, 'reg_alpha': 0.01, 'reg_lambda': 8.0, 'subsample': 0.85, 'colsample_bytree': 0.85, 'num_leaves': 31, 'min_child_samples': 80}. Best is trial 0 with value: 0.9417468412110894.


  lgb/interaction fold=0 AUC=0.941924
  lgb/interaction fold=1 AUC=0.940814
  lgb/interaction fold=2 AUC=0.941726
  lgb/interaction fold=3 AUC=0.941986


[I 2026-09-09 15:09:51,792] Trial 1 finished with value: 0.9416021346940467 and parameters: {'variant': 'interaction', 'learning_rate': 0.04, 'max_depth': 6, 'reg_alpha': 0.01, 'reg_lambda': 8.0, 'subsample': 0.85, 'colsample_bytree': 0.85, 'num_leaves': 31, 'min_child_samples': 80}. Best is trial 0 with value: 0.9417468412110894.
[I 2026-09-09 15:09:51,856] A new study created in RDB with name: xgb


  xgb/raw fold=0 AUC=0.942143
  xgb/raw fold=1 AUC=0.940933
  xgb/raw fold=2 AUC=0.941926
  xgb/raw fold=3 AUC=0.942241


[I 2026-09-09 15:12:40,577] Trial 0 finished with value: 0.9418043229193733 and parameters: {'variant': 'raw', 'learning_rate': 0.04, 'max_depth': 5, 'reg_alpha': 0.01, 'reg_lambda': 8.0, 'subsample': 0.85, 'colsample_bytree': 0.85, 'min_child_weight': 8.0, 'gamma': 0.1}. Best is trial 0 with value: 0.9418043229193733.


  xgb/interaction fold=0 AUC=0.941916
  xgb/interaction fold=1 AUC=0.940841
  xgb/interaction fold=2 AUC=0.941695
  xgb/interaction fold=3 AUC=0.941948
Search complete. Sealed scores have NOT been computed.


[I 2026-09-09 15:16:19,624] Trial 1 finished with value: 0.9415924464603173 and parameters: {'variant': 'interaction', 'learning_rate': 0.04, 'max_depth': 5, 'reg_alpha': 0.01, 'reg_lambda': 8.0, 'subsample': 0.85, 'colsample_bytree': 0.85, 'min_child_weight': 8.0, 'gamma': 0.1}. Best is trial 0 with value: 0.9418043229193733.


## 候補・seed平均・重みを固定
この段階以降は同じrunへの追加探索をロックします。探索予算を増やす場合このセルの前に実施します。

In [11]:
execute('freeze')


Already frozen; no changes made.


## 封印評価
一度だけ独立holdoutを評価し、固定LightGBMとのpaired bootstrap差分も記録します。Public LBとの直接比較には使いません。

In [12]:
execute('audit')


{
  "sealed_auc": 0.9418656926391448,
  "baseline_sealed_auc": 0.941704579526343,
  "public_lb": null,
  "public_lb_target": 0.94635,
  "sealed_rows": 133733,
  "note": "Public LB is unmeasured. Do not tune using this sealed result.",
  "delta_vs_baseline": 0.00016111311280175844,
  "auc_95_ci": [
    0.9405083811212422,
    0.9432175058186937
  ],
  "delta_95_ci": [
    9.448479904459184e-05,
    0.00022626686244050774
  ],
  "bootstrap_replicates": 300
}


## 最終5-fold・提出用CSV
選択後の最終OOFは独立スコアではありません。提出用はsample_submissionのID順に揃えます。

In [ ]:
execute('finalize')
import json
import pandas as pd
from IPython.display import FileLink, display
display(pd.read_csv(RUN / 'submission.csv').head())
display(FileLink(str(RUN / 'submission.csv')))
print(json.loads((RUN / 'sealed_report.json').read_text()))


Final candidate=0, fold=0 done
Final candidate=0, fold=1 done
Final candidate=0, fold=2 done
Final candidate=0, fold=3 done
Final candidate=0, fold=4 done
Final candidate=1, fold=0 done
Final candidate=1, fold=1 done
Final candidate=1, fold=2 done
